# Grupo C · Favelas e comunidades urbanas e equipamentos públicos
**Oficina Python para Mapeamento Urbano** · Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/))

O Censo 2022 identifica os setores censitários localizados em **Favelas e Comunidades Urbanas** (coluna `CD_FCU`). Vocês vão medir a distância desses setores até escolas e unidades de saúde mapeadas no OpenStreetMap e comparar com o restante da cidade.

### O que o grupo entrega (60 min)

| Tempo | Etapa | Produto |
|---|---|---|
| 10 min | Pergunta e hipótese | H1 e H0 escritas na seção 1 |
| 15 min | Conhecer e limpar o dado | tipos corrigidos, vazios tratados, recorte definido |
| 20 min | Um método **espacial** + um método **quantitativo** | tabela de resultados |
| 15 min | Mapa + gráfico com créditos | `outputs/grupo_X_mapa.png` e `outputs/grupo_X_grafico.png` |

No fim, cada grupo apresenta em 6 minutos: pergunta, método, mapa, gráfico e **um limite** da análise.

## 0. Ambiente

In [ ]:
!pip install -q geobr osmnx mapclassify folium
import os, sys
REPO = "https://github.com/brfelipealmeida/oficina-python-mapeamento"
if not os.path.exists("oficina-python-mapeamento"):
    !git clone -q {REPO}
sys.path.append("oficina-python-mapeamento/src")

import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from estilo import *
from dados import *
configurar_estilo()

## 1. Pergunta e hipótese

Escolham **uma** e reescrevam (ou criem outra):

- Setores em favelas e comunidades urbanas estão mais longe de unidades de saúde?
- Há diferença no número de escolas a até 1 km?
- A diferença some quando controlamos pela densidade?

**H1:** _escreva aqui_
**H0:** _escreva aqui_
**Variável dependente:** _..._  **Variável independente:** _..._

## 2. Carregar os dados (pronto)

In [ ]:
setores = setores_bh()
BH = "Belo Horizonte, Minas Gerais, Brazil"
escolas = osm_feicoes({"amenity": ["school"]}, lugar=BH)
saude = osm_feicoes({"amenity": ["clinic", "hospital", "doctors"], "healthcare": True}, lugar=BH)
for g in (escolas, saude):
    g["geometry"] = g.geometry.centroid
print(len(setores), "setores |", len(escolas), "escolas |", len(saude), "equipamentos de saúde")
print(setores["em_favela"].value_counts())

## 3. Conhecer e limpar

1. Como a coluna `em_favela` foi criada a partir de `CD_FCU`? (veja `src/dados.py`)
2. Quais favelas aparecem em `NM_FCU`? Quantos setores e moradores cada uma tem?
3. O OSM tem **todas** as escolas e unidades de saúde? Como poderíamos checar? (dica: a Prefeitura publica a lista oficial no portal de dados abertos)
4. Há equipamentos duplicados (o mesmo hospital como ponto e como polígono)?

In [ ]:
setores[setores.em_favela].groupby("NM_FCU")["pessoas"].agg(["count", "sum"]).sort_values("sum", ascending=False).head(10)

In [ ]:
# TODO: limpar duplicados de equipamentos (mesmo nome a menos de 50 m) e setores sem população
validos = setores[setores["pessoas"] > 0].copy()
validos["dens_ha"] = validos["pessoas"] / (validos.area / 10_000)

## 4. Método espacial (escolha um e complete)

**a) Distância ao equipamento mais próximo** (`sjoin_nearest`).
**b) Quantos equipamentos a até 1 km** (`buffer` + `sjoin` + contagem).

In [ ]:
cent = validos[["CD_SETOR", "geometry"]].copy()
cent["geometry"] = validos.centroid
for nome, pts in (("saude", saude), ("escola", escolas)):
    p = gpd.sjoin_nearest(cent, pts[["geometry"]], distance_col=f"dist_{nome}_m")
    validos[f"dist_{nome}_m"] = validos["CD_SETOR"].map(p.groupby("CD_SETOR")[f"dist_{nome}_m"].min())

validos.groupby("em_favela")[["dist_saude_m", "dist_escola_m"]].median().round(0)

In [ ]:
# b) TODO: contagem a até 1 km
# raio = cent.copy(); raio["geometry"] = cent.buffer(1000)
# cont = gpd.sjoin(raio, escolas[["geometry"]], predicate="contains").groupby("CD_SETOR").size()
# ...

## 5. Método quantitativo (escolha um)

| Opção | Quando usar | Código |
|---|---|---|
| Correlação | duas variáveis numéricas | `df["a"].corr(df["b"], method="spearman")` |
| Regressão | quanto Y muda quando X muda, com controles | `smf.ols("y ~ x + controle", data=df).fit()` |
| Comparação de grupos | uma variável numérica entre categorias | `stats.mannwhitneyu(g1, g2)` |

Lembre: p-valor pequeno indica que a diferença dificilmente é acaso; **não** indica que ela é grande ou importante. Olhe também o tamanho do efeito (diferença de medianas, coeficiente).

In [ ]:
fav = validos.loc[validos.em_favela, "dist_saude_m"]
resto = validos.loc[~validos.em_favela, "dist_saude_m"]
print("Medianas (m):", round(fav.median()), "em FCU vs", round(resto.median()), "no restante")
print(stats.mannwhitneyu(fav, resto))

modelo = smf.ols("np.log1p(dist_saude_m) ~ em_favela + np.log1p(dens_ha)", data=validos).fit()
print(modelo.summary().tables[1])

**Atenção à leitura:** distância em linha reta não é acesso. Em BH, muitas favelas estão em encostas: 400 m em linha reta podem ser 15 minutos de escada. Uma extensão possível é refazer a distância pela rede de ruas com `osmnx` (como na rota A→B do live coding).

## 6. Mapa

In [ ]:
fig, ax = mapa_base((9, 10))
validos.plot(ax=ax, column="dist_saude_m", scheme="quantiles", k=5, cmap=CMAP, linewidth=0,
             legend=True, legend_kwds={"title": "metros até a saúde", "fmt": "{:.0f}", "loc": "lower right"})
validos[validos.em_favela].dissolve().boundary.plot(ax=ax, color=CORES["minerio"], linewidth=0.8)
saude.plot(ax=ax, color=CORES["tinta"], markersize=3)
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, "TODO: título que responde à pergunta", "Contorno: setores em favelas e comunidades urbanas")
creditos(fig, fonte="IBGE, Censo 2022; OpenStreetMap", nota="Grupo C")
salvar(fig, "grupo_C_mapa")

## 7. Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([resto, fav], tick_labels=["Demais setores", "Favelas e comunidades"], patch_artist=True,
           boxprops={"facecolor": CORES["lilas"]}, medianprops={"color": CORES["roxo"]}, showfliers=False)
ax.set_ylabel("Distância à unidade de saúde mais próxima (m)")
titulo(ax, "TODO: título", "Setores censitários de BH, 2022")
creditos(fig, fonte="IBGE, Censo 2022; OpenStreetMap", nota="Grupo C")
salvar(fig, "grupo_C_grafico")

## 8. Conclusão (escreva aqui)

1. **Resultado:** a hipótese foi sustentada, refutada ou ficou inconclusiva? Com que número?
2. **Leitura urbana:** o que isso significa para quem planeja a cidade?
3. **Limite:** o que o dado não mostra? (cobertura do OSM, escala do setor, distância em linha reta vs. caminho real, relevo...)